# Optimizing Facial Recognition: An Analytical Comparison of Traditional and Deep Learning Approaches

Code for the paper:

> Shadman Sakib, Md Nurul Absar Siddiky, **Md. Arifuzzaman**, Md. Jalal Uddin Chowdhury.
> *Optimizing Facial Recognition: An Analytical Comparison of Traditional and Deep Learning Approaches.*
> 2024 International Conference on Data Science and Its Applications (ICoDSA), IEEE, pp. 271–276.
> [doi:10.1109/ICoDSA62899.2024.10651632](https://doi.org/10.1109/ICoDSA62899.2024.10651632)

Methods compared on the **Olivetti (ORL / AT&T) faces** dataset (400 images, 40 people, 64 × 64 grey-scale):

1. **Eigenfaces** — PCA + nearest neighbour in face space (300 train / 100 test)
2. **SVM** — RBF, linear and polynomial kernels across C values (80 / 20 split)
3. **Neural network** — 3 dense layers (256-128-128) with dropout (80 / 20 split)
4. **Transfer learning** — frozen ImageNet MobileNet + new classification head (80 / 20 split)

**Run on Google Colab:** Runtime → Change runtime type → **GPU** (recommended for step 4) → Run all.

## 1. Setup

In [ ]:
import os, random, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
os.makedirs("figures", exist_ok=True)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Load the Olivetti faces dataset

400 grey-scale images (64 × 64, values in [0, 1]) of **40 people, 10 images each**, taken at AT&T Laboratories Cambridge between 1992 and 1994 with varying lighting, expression and details (glasses / no glasses).

The arrays are stored in `data/`. If they are missing, the notebook downloads them from this repository, or from scikit-learn as a fallback.

In [ ]:
IMG_PATH, TGT_PATH = "data/olivetti_faces.npy", "data/olivetti_faces_target.npy"
BASE_URL = "https://raw.githubusercontent.com/arifex/Face-Recognition-PCA-SVM-MobileNet/main/data/"

if not (os.path.exists(IMG_PATH) and os.path.exists(TGT_PATH)):
    os.makedirs("data", exist_ok=True)
    try:
        import urllib.request
        for f in ["olivetti_faces.npy", "olivetti_faces_target.npy"]:
            urllib.request.urlretrieve(BASE_URL + f, "data/" + f)
    except Exception:
        from sklearn.datasets import fetch_olivetti_faces
        d = fetch_olivetti_faces()
        np.save(IMG_PATH, d.images.astype("float32")); np.save(TGT_PATH, d.target)

images = np.load(IMG_PATH)            # (400, 64, 64)
targets = np.load(TGT_PATH)           # (400,)
X = images.reshape(len(images), -1)   # (400, 4096) flattened
y = targets
n_classes = len(np.unique(y))
print("Images:", images.shape, "| classes:", n_classes, "| per class:", np.bincount(y)[0])

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(14, 6))
for person, ax in enumerate(axes.ravel()):
    ax.imshow(images[np.where(y == person)[0][0]], cmap="gray"); ax.set_title(f"#{person}", fontsize=9); ax.axis("off")
plt.suptitle("One image of each of the 40 subjects"); plt.tight_layout()
plt.savefig("figures/sample_faces.png", dpi=120); plt.show()

## 3. Eigenfaces (PCA)

Given training faces Γ₁ … Γ_M:

* mean face: ψ = (1/M) Σ Γₙ
* centred faces: φᵢ = Γᵢ − ψ
* covariance: C = (1/M) Σ φₙ φₙᵀ, an N² × N² matrix (4096 × 4096)

C is too large to decompose directly, so we use the standard trick: take the eigenvectors **v** of the small M × M matrix AAᵀ (A = rows φᵢ). Then **u** = Aᵀ**v** are eigenvectors of C, the *eigenfaces*.

A face is projected into face space as w_k = u_kᵀ (Γ − ψ) and classified by the **nearest training face (Euclidean distance)** in that space. Accuracy is measured for K = 1 … 80 eigenfaces.

In [ ]:
Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(X, y, test_size=100, random_state=SEED)
print("Train:", Xtr_e.shape, "Test:", Xte_e.shape)

psi = Xtr_e.mean(axis=0)                       # mean face
A = Xtr_e - psi                                # centred training faces (M x N^2)
eigvals, V = np.linalg.eigh(A @ A.T)           # M x M problem
order = np.argsort(eigvals)[::-1]
eigvals, V = eigvals[order], V[:, order]
U = A.T @ V                                    # eigenfaces (N^2 x M)
U /= np.linalg.norm(U, axis=0) + 1e-12

def eigenface_accuracy(K):
    W_train = A @ U[:, :K]
    W_test = (Xte_e - psi) @ U[:, :K]
    dists = ((W_test[:, None, :] - W_train[None, :, :]) ** 2).sum(-1)
    pred = ytr_e[dists.argmin(axis=1)]
    return accuracy_score(yte_e, pred), pred

Ks = list(range(1, 81))
eig_acc = [eigenface_accuracy(k)[0] for k in Ks]
best_K = Ks[int(np.argmax(eig_acc))]
print(f"Best accuracy {100*max(eig_acc):.2f}% at K = {best_K}  |  K = 16: {100*eig_acc[15]:.2f}%  |  K = 80: {100*eig_acc[-1]:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(Ks, np.array(eig_acc) * 100, marker=".", color="#1f6f7f")
axes[0].axvline(best_K, ls="--", color="#a9521a", label=f"best K = {best_K}")
axes[0].set_xlabel("Number of eigenfaces (K)"); axes[0].set_ylabel("Test accuracy (%)")
axes[0].set_title("Eigenfaces: accuracy vs number of principal components"); axes[0].legend()
explained = np.cumsum(eigvals) / eigvals.sum()
axes[1].plot(range(1, len(explained) + 1), explained * 100, color="#1f6f7f")
axes[1].set_xlabel("Number of eigenfaces"); axes[1].set_ylabel("Cumulative variance (%)"); axes[1].set_title("Variance explained")
plt.tight_layout(); plt.savefig("figures/eigenface_accuracy.png", dpi=150); plt.show()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
axes[0, 0].imshow(psi.reshape(64, 64), cmap="gray"); axes[0, 0].set_title("mean face")
for i, ax in enumerate(axes.ravel()[1:]):
    ax.imshow(U[:, i].reshape(64, 64), cmap="gray"); ax.set_title(f"eigenface {i+1}", fontsize=9)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.savefig("figures/eigenfaces.png", dpi=120); plt.show()

## 4. Support Vector Machine

80 / 20 split (320 train / 80 test). A linear SVM finds the maximum-margin hyperplane w·x + b = 0 by minimising ½‖w‖² + C Σ ξᵢ. Multi-class recognition uses **one-against-one** (scikit-learn's default for `SVC`).

C is swept from 10⁻³ to 10³ for the RBF (`gamma='scale'`), linear and polynomial kernels.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Train:", Xtr.shape, "Test:", Xte.shape)

C_values = [1e-3, 1e-2, 1e-1, 1, 10, 100, 1000]
svm_results = {}
for kernel in ["rbf", "linear", "poly"]:
    svm_results[kernel] = [accuracy_score(yte, SVC(kernel=kernel, C=c, gamma="scale").fit(Xtr, ytr).predict(Xte))
                           for c in C_values]
svm_table = pd.DataFrame(svm_results, index=[f"C={c:g}" for c in C_values]).mul(100).round(2)
display(svm_table)

plt.figure(figsize=(8, 4.5))
for kernel, marker in zip(svm_results, ["o", "s", "^"]):
    plt.semilogx(C_values, np.array(svm_results[kernel]) * 100, marker=marker, label=kernel)
plt.xlabel("C (log scale)"); plt.ylabel("Test accuracy (%)"); plt.title("SVM: accuracy vs C for different kernels")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("figures/svm_kernels.png", dpi=150); plt.show()

best_svm = {k: max(v) for k, v in svm_results.items()}
print("Best accuracy per kernel:", {k: f"{100*v:.2f}%" for k, v in best_svm.items()})

In [ ]:
svm_linear = SVC(kernel="linear", C=1).fit(Xtr, ytr)
pred_svm = svm_linear.predict(Xte)
print(f"Linear SVM test accuracy: {100*accuracy_score(yte, pred_svm):.2f}%\n")
print(classification_report(yte, pred_svm, zero_division=0))

## 5. Neural network (three dense layers)

The architecture described in the paper: a Keras `Sequential` model on the flattened 4,096-pixel input with **Dense(256) → Dropout(0.2) → Dense(128) → Dropout(0.2) → Dense(128) → Dropout(0.2) → Dense(40, softmax)**. It is trained with Adam (lr = 0.001) and categorical cross-entropy for 100 epochs, batch size 128, with `ReduceLROnPlateau`.

*Note:* following the paper's setup, the 80 test images also serve as Keras validation data, so the learning-rate schedule sees test loss. A held-out validation split would be stricter.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
ytr_oh, yte_oh = keras.utils.to_categorical(ytr, n_classes), keras.utils.to_categorical(yte, n_classes)

mlp = keras.Sequential([
    layers.Input(shape=(X.shape[1],)),
    layers.Dense(256, activation="relu"), layers.Dropout(0.2),
    layers.Dense(128, activation="relu"), layers.Dropout(0.2),
    layers.Dense(128, activation="relu"), layers.Dropout(0.2),
    layers.Dense(n_classes, activation="softmax"),
])
mlp.compile(optimizer=keras.optimizers.Adam(0.001), loss="categorical_crossentropy", metrics=["accuracy"])
mlp.summary()

hist_mlp = mlp.fit(Xtr, ytr_oh, validation_data=(Xte, yte_oh), epochs=100, batch_size=128, verbose=0,
                   callbacks=[keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-5)])
mlp_acc = mlp.evaluate(Xte, yte_oh, verbose=0)[1]
print(f"Neural network test accuracy: {100*mlp_acc:.2f}%")

## 6. Transfer learning with MobileNet

Each grey-scale face is copied into 3 channels, resized to 224 × 224 and scaled to [−1, 1] (MobileNet's input format). The **ImageNet-pretrained MobileNet base is frozen**, and a new head is trained on top: **GlobalAveragePooling2D → Dense(1024, ReLU) → Dense(40, softmax)**. Training uses Adam (lr = 0.0004) and categorical cross-entropy for 100 epochs, batch size 64.

*A GPU runtime takes about 2–4 minutes for this step; CPU is much slower.*

In [ ]:
def to_mobilenet_input(flat):
    img = flat.reshape(-1, 64, 64, 1)
    img = np.repeat(img, 3, axis=-1)                     # grey -> 3 channels
    img = tf.image.resize(img, (224, 224)).numpy()
    return img * 2.0 - 1.0                               # [0,1] -> [-1,1]

Xtr_tl, Xte_tl = to_mobilenet_input(Xtr), to_mobilenet_input(Xte)

tf.keras.utils.set_random_seed(SEED)
base = keras.applications.MobileNet(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base.trainable = False                                   # freeze pretrained layers

tl_model = keras.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1024, activation="relu"),
    layers.Dense(n_classes, activation="softmax"),
])
tl_model.compile(optimizer=keras.optimizers.Adam(0.0004), loss="categorical_crossentropy", metrics=["accuracy"])

hist_tl = tl_model.fit(Xtr_tl, ytr_oh, validation_data=(Xte_tl, yte_oh), epochs=100, batch_size=64, verbose=0)
pred_tl = tl_model.predict(Xte_tl, verbose=0).argmax(axis=1)
tl_acc = accuracy_score(yte, pred_tl)
print(f"MobileNet transfer learning test accuracy: {100*tl_acc:.2f}%\n")
print(classification_report(yte, pred_tl, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, h, title in [(axes[0], hist_mlp, "Neural network (dense)"), (axes[1], hist_tl, "MobileNet transfer learning")]:
    ax.plot(h.history["accuracy"], label="train"); ax.plot(h.history["val_accuracy"], label="test")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.set_title(title); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("figures/training_curves.png", dpi=150); plt.show()

## 7. Robustness check — 5-fold cross-validation

With only 80 test images, one misclassified face changes accuracy by 1.25 points, so a single split can be lucky or unlucky. Here the two classical methods are re-evaluated with **stratified 5-fold cross-validation** (every image is tested exactly once).

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = {}
for kernel in ["linear", "poly", "rbf"]:
    s = cross_val_score(SVC(kernel=kernel, C=10 if kernel == "rbf" else 1, gamma="scale"), X, y, cv=cv5)
    cv_rows[f"SVM ({kernel})"] = s

def eigenface_cv(K):
    accs = []
    for tr, te in cv5.split(X, y):
        mu = X[tr].mean(0); A_ = X[tr] - mu
        w, V_ = np.linalg.eigh(A_ @ A_.T); V_ = V_[:, np.argsort(w)[::-1]]
        U_ = A_.T @ V_; U_ /= np.linalg.norm(U_, axis=0) + 1e-12
        Wtr, Wte = A_ @ U_[:, :K], (X[te] - mu) @ U_[:, :K]
        pred = y[tr][((Wte[:, None] - Wtr[None]) ** 2).sum(-1).argmin(1)]
        accs.append(accuracy_score(y[te], pred))
    return np.array(accs)

cv_rows[f"Eigenfaces (K={best_K})"] = eigenface_cv(best_K)
cv_table = pd.DataFrame({k: {"Mean accuracy (%)": round(100*v.mean(), 2), "Std (%)": round(100*v.std(), 2)}
                         for k, v in cv_rows.items()}).T
display(cv_table)

## 8. Summary

In [ ]:
summary = pd.DataFrame({
    "Method": ["Eigenfaces (PCA + nearest neighbour)", "SVM (RBF)", "SVM (linear)", "SVM (polynomial)",
               "Neural network (3 dense layers)", "Transfer learning (MobileNet)"],
    "Test accuracy (%)": [100*max(eig_acc), 100*best_svm["rbf"], 100*best_svm["linear"], 100*best_svm["poly"],
                          100*mlp_acc, 100*tl_acc],
}).round(2)
summary

**Takeaways**

* On this small dataset (10 images per person), classical methods such as linear and polynomial SVMs on raw pixels and eigenfaces match or beat deep models. The data is close to linearly separable, and the strong inductive bias suits small samples.
* A dense network trained from scratch has too many parameters for 320 training images and is the weakest method.
* Transfer learning from ImageNet recovers most of the gap without training a deep network from scratch.
* Single-split accuracies move in steps of 1.25 points (80 test images), so the 5-fold CV figures are more reliable for comparing methods.